# FOVL Assignment — Multilingual Word Embeddings for Indian Languages

## Roll Number: M26AI1159 | Sarthak B. Narnor

**Approach:** one multilingual **FastText** model (300-d, character n-gram subwords), trained from scratch on IndicCorpV2 only.
FastText gives a vector for *any* string by summing its character n-gram vectors, which is our OOV strategy (no word pair is ever dropped).

**Rules respected:** no pre-trained vectors, no external corpora; the human similarity scores are used only to *measure* Spearman (and to choose between a handful of hyper-parameter settings) — never as a training signal.

In [1]:
# ---------------- Setup ----------------
import os, re, sys, json, zipfile, random, unicodedata, subprocess, shutil, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

def ensure(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for p, i in [("gensim",None),("datasets",None),("python-docx","docx"),("scikit-learn","sklearn"),("scipy",None),("matplotlib",None)]:
    ensure(p, i)

from gensim.models import FastText, Word2Vec
from scipy.stats import spearmanr

# ---------------- Config (edit these) ----------------
STUDENT_NAME = "Sarthak B. Narnor"          # shown on the report title page
ROLL_NO      = "M26AI1159"
SMOKE   = os.environ.get("FOVL_SMOKE") == "1"   # synthetic data, only used to test the code path
SEED, DIM = 42, 300                  # DIM must be exactly 300
HF_ID    = "rony000013/indiccorpV2-5k"
BENCH_URL = "https://github.com/syedsarfarazakhtar/Word-Similarity-Datasets-for-Indian-Languages.git"
WORK = Path("fovl_work"); OUT = WORK/"submission"
INCLUDE_MODEL_IN_ZIP = False         # model can be several hundred MB; set True if your portal allows it
for d in (WORK, OUT, OUT/"embeddings", OUT/"vocab", OUT/"figures", OUT/"model"): d.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
NCPU = max(1, os.cpu_count() or 1)
print("smoke mode:", SMOKE, "| cpus:", NCPU)

smoke mode: False | cpus: 2


## 1. Text normalisation (shared by corpus and benchmark words)

In [2]:
PUNCT = re.compile(r"[।॥۔،؛؟‘’“”–—…«»•"
                   r"!\"#$%&'()*+,\-./:;<=>?@\[\\\]^_`{|}~"
                   r"0-9०-९০-৯੦-੯૦-૯୦-୯௦-௯౦-౯೦-೯൦-൯۰-۹٠-٩]+")
ZW = re.compile("[​‌‍⁠﻿]")
SENT_SPLIT = re.compile(r"[\n।॥۔.?!]+")

def norm(text):
    return ZW.sub("", unicodedata.normalize("NFC", str(text))).lower()

def tokenize(text):
    return [t for t in PUNCT.sub(" ", norm(text)).split() if t]

def sentences_from_text(text):
    out = []
    for seg in SENT_SPLIT.split(norm(text)):
        toks = PUNCT.sub(" ", seg).split()
        if len(toks) >= 2: out.append(toks)
    return out

print(tokenize("यह एक परीक्षण वाक्य है। 123"))

['यह', 'एक', 'परीक्षण', 'वाक्य', 'है']


## 2. Data: IndicCorpV2 (training) and the Akhtar et al. (2017) benchmark (evaluation only)

In [3]:
LANG_CODES = {"hindi":"hi","urdu":"ur","bengali":"bn","bangla":"bn","telugu":"te","marathi":"mr","punjabi":"pa","panjabi":"pa",
              "gujarati":"gu","tamil":"ta","malayalam":"ml","kannada":"kn","odia":"or","oriya":"or","assamese":"as",
              "nepali":"ne","sanskrit":"sa","sindhi":"sd","konkani":"kok","maithili":"mai","dogri":"doi","kashmiri":"ks",
              "bodo":"brx","manipuri":"mni","santali":"sat","english":"en"}
CODE_SET = set(LANG_CODES.values())

def to_code(x):
    x = str(x).strip().lower()
    if x in LANG_CODES: return LANG_CODES[x]
    if x in CODE_SET: return x
    x2 = re.split(r"[_\-]", x)[0]
    return LANG_CODES.get(x2, x2 if x2 in CODE_SET else None)

def guess_code_from_path(p):
    for part in [p.stem] + list(p.parts[::-1]):
        low = part.lower()
        for name, c in LANG_CODES.items():
            if name in low: return c
        c = to_code(low)
        if c: return c
    return None

# ---------- corpus ----------
def synthetic_world():
    rng = random.Random(SEED)
    syl = {"hi":"कखगचजतदनपबमयरलवसह","bn":"কখগচজতদনপবমযরলশসহ","ta":"கசதநபமயரலவ"}
    vow = {"hi":["ा","ि","ी","ु","े","ो",""],"bn":["া","ি","ী","ু","ে","ো",""],"ta":["ா","ி","ீ","ு","ெ","ொ",""]}
    corpus, bench = [], {}
    for lg in syl:
        roots = ["".join(rng.choice(syl[lg])+rng.choice(vow[lg]) for _ in range(rng.randint(1,3))) for _ in range(120)]
        sufs = ["", "ों", "ें", "ता", "ना"] if lg=="hi" else ["", "রা", "কে", "তা"] if lg=="bn" else ["", "கள்", "ின்", "ை"]
        for _ in range(3000):
            k = rng.randrange(0, 120, 1); L = rng.randint(5, 12)
            near = [roots[(k+rng.randint(-2,2)) % 120] + rng.choice(sufs) for _ in range(L)]
            corpus.append((lg, " ".join(near) + " ।"))
        pairs = []
        for _ in range(150):
            a = rng.randrange(120); b = (a + rng.randint(-15, 15)) % 120
            pairs.append((roots[a]+rng.choice(sufs), roots[b]+rng.choice(sufs), round(10 - abs(a-b)*10/16 + rng.random(), 2)))
        bench[lg] = pairs
    return corpus, bench

def load_corpus():
    if SMOKE: return pd.DataFrame(synthetic_world()[0], columns=["lang","text"])
    from datasets import load_dataset, get_dataset_config_names
    try:
        ds = load_dataset(HF_ID)
        df = pd.concat([ds[s].to_pandas() for s in ds], ignore_index=True)
    except Exception as e:
        print("default load failed (", repr(e)[:120], ") -> trying per-config loading")
        frames = []
        for cfg in get_dataset_config_names(HF_ID):
            d = load_dataset(HF_ID, cfg)
            part = pd.concat([d[s].to_pandas() for s in d], ignore_index=True); part["lang"] = cfg; frames.append(part)
        df = pd.concat(frames, ignore_index=True)
    str_cols = [c for c in df.columns if df[c].map(lambda v: isinstance(v, str)).mean() > 0.9]
    text_col = "text" if "text" in df.columns else max(str_cols, key=lambda c: df[c].str.len().mean())
    lang_col = next((c for c in ["lang","language","Language","lang_code","config"] if c in df.columns), None)
    if lang_col is None:
        lang_col = next((c for c in str_cols if c != text_col and df[c].nunique() <= 60), None)
    print("columns:", list(df.columns), "| text column:", text_col, "| language column:", lang_col)
    out = pd.DataFrame({"text": df[text_col].astype(str)})
    out["lang"] = df[lang_col].map(to_code) if lang_col else None
    return out

corpus_df = load_corpus()
print(corpus_df.shape); print(corpus_df.head(3))
if corpus_df["lang"].notna().any(): print(corpus_df["lang"].value_counts().to_string())

README.md:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

data.parquet: reconstructing file:   0%|          |  0.00B / 28.8MB            

data.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/118243 [00:00<?, ? examples/s]

columns: ['language', 'code', 'text'] | text column: text | language column: language
(118243, 2)
                                                text lang
0  ঘটনাৰ বিৱৰণি মতে বোকাকাতৰ কৈলাখাতত নিয়মিতভাৱে ...   as
1  ৰাণু মণ্ডলৰ সুৱদী কন্ঠত মুগ্ধ হৈ ভাইজান ছলমানে...   as
2  ইফালে আজিৰ মেচখন সম্পৰ্কে নৰ্থ-ইষ্টৰ প্ৰশিক্ষক...   as
lang
as     5000
brx    5000
bn     5000
doi    5000
en     5000
gu     5000
hi     5000
kn     5000
mai    5000
ml     5000
mr     5000
mni    5000
ne     5000
or     5000
sd     5000
pa     5000
sa     5000
sat    5000
te     5000
ta     5000
ur     5000
ks     3243


In [4]:
# ---------- benchmark ----------
def parse_pairs(p):
    rows = []
    for line in p.read_text(encoding="utf-8-sig", errors="replace").splitlines():
        line = line.strip()
        if not line: continue
        parts = None
        for sep in ("\t", ";", ","):
            if sep in line: parts = [x.strip() for x in line.split(sep)]; break
        if parts is None: parts = line.split()
        if len(parts) < 3: continue
        try: s = float(parts[-1])
        except ValueError: continue                      # header / junk line
        if len(parts) >= 4 and re.fullmatch(r"\d+", parts[0]): parts = parts[1:]   # leading index column
        if parts[0] and parts[1]: rows.append((parts[0], parts[1], s))
    return rows

def load_benchmark():
    if SMOKE: return synthetic_world()[1], {k: k for k in synthetic_world()[1]}
    root = WORK/"bench"
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", BENCH_URL, str(root)], check=True)
    bench, codes, seen = {}, {}, []
    for p in sorted(root.rglob("*")):
        if ".git" in p.parts or not p.is_file(): continue
        if p.suffix.lower() not in {".txt", ".csv", ".tsv"} or p.name.lower().startswith(("readme", "license")): continue
        rows = parse_pairs(p)
        if len(rows) >= 20:
            key = p.stem; bench[key] = rows; codes[key] = guess_code_from_path(p.relative_to(root)); seen.append((str(p.relative_to(root)), len(rows), codes[key]))
    print(pd.DataFrame(seen, columns=["file", "pairs", "corpus lang code"]).to_string(index=False))
    return bench, codes

BENCH, BENCH_CODE = load_benchmark()
BENCH_CODE = BENCH_CODE if isinstance(BENCH_CODE, dict) else {}
print("\nbenchmark languages:", {k: len(v) for k, v in BENCH.items()})
# >>> CHECK the table above: every file should be one language with a sensible pair count and corpus-language code.

           file  pairs corpus lang code
Gujarati-WS.txt    163               gu
   Hindi-WS.txt    235               hi
 Marathi-WS.txt    104               mr
 Punjabi-WS.txt    143               pa
   Tamil-WS.txt     96               ta
  Telugu-WS.txt    111               te
    Urdu-WS.txt    100               ur

benchmark languages: {'Gujarati-WS': 163, 'Hindi-WS': 235, 'Marathi-WS': 104, 'Punjabi-WS': 143, 'Tamil-WS': 96, 'Telugu-WS': 111, 'Urdu-WS': 100}


## 3. Corpus -> sentences

In [5]:
SENTS_BY_LANG, ALL_SENTS = {}, []
for lang, text in zip(corpus_df["lang"], corpus_df["text"]):
    ss = sentences_from_text(text)
    ALL_SENTS.extend(ss)
    if lang: SENTS_BY_LANG.setdefault(lang, []).extend(ss)
tok_total = sum(map(len, ALL_SENTS))
print(f"{len(ALL_SENTS):,} sentences, {tok_total:,} tokens")
corpus_stats = pd.DataFrame({"sentences": {k: len(v) for k, v in SENTS_BY_LANG.items()},
                             "tokens": {k: sum(map(len, v)) for k, v in SENTS_BY_LANG.items()}})
corpus_stats

329,346 sentences, 4,396,950 tokens


,sentences,tokens
as,11605,201508
brx,8874,142103
bn,19098,230087
doi,7115,85706
en,10380,187219
gu,14630,201483
hi,16404,281207
kn,13116,127911
ks,5171,62627
mai,11456,138386


## 4. Model, OOV strategy and evaluation

* **Model:** gensim `FastText`, `vector_size=300`. A word vector = mean of the vectors of the word itself and its character n-grams (`min_n..max_n`, hashed into `bucket` slots).
* **OOV strategy:** for a word not in the vocabulary we sum its character n-gram vectors, so morphologically related Indic word forms (inflections, suffixes) land close together. If a benchmark entry contains several tokens, we average their vectors. No pair is discarded.
* **Metric:** cosine similarity per pair, then Spearman’s rho against the human scores, per language, and the macro average.

In [6]:
class Embedder:
    # holds either one joint model, or {key: model}; oov_zero=True mimics a word-level model (OOV -> zero vector)
    def __init__(self, models, oov_zero=False): self.models, self.oov_zero = models, oov_zero
    def _m(self, key): return self.models if not isinstance(self.models, dict) else self.models.get(key, next(iter(self.models.values())))
    def in_vocab(self, w, key=None):
        m = self._m(key); toks = tokenize(w) or [w]
        return all(t in m.wv.key_to_index for t in toks)
    def vec(self, w, key=None):
        m = self._m(key); toks = tokenize(w) or [norm(w)]
        vs = []
        for t in toks:
            if self.oov_zero and t not in m.wv.key_to_index: vs.append(np.zeros(DIM, np.float32))
            else: vs.append(m.wv[t])
        return np.mean(vs, axis=0).astype(np.float32)
    def save(self, d):
        d = Path(d); d.mkdir(parents=True, exist_ok=True)
        if isinstance(self.models, dict):
            for k, m in self.models.items(): m.save(str(d/f"{k}.model"))
            json.dump({"per_language": True, "keys": list(self.models)}, open(d/"meta.json", "w"))
        else:
            self.models.save(str(d/"joint.model")); json.dump({"per_language": False}, open(d/"meta.json", "w"))

def cos(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(a @ b / (na * nb)) if na > 0 and nb > 0 else 0.0

def evaluate(emb):
    res = {}
    for key, rows in BENCH.items():
        sims = np.array([cos(emb.vec(a, key), emb.vec(b, key)) for a, b, _ in rows])
        gold = np.array([s for _, _, s in rows])
        words = [w for a, b, _ in rows for w in (a, b)]
        flags = [emb.in_vocab(w, key) for w in words]
        pair_in = np.array(flags).reshape(-1, 2)
        uw = list(dict.fromkeys(words)); uflag = [emb.in_vocab(w, key) for w in uw]
        res[key] = dict(rho=float(spearmanr(sims, gold).correlation), n_pairs=len(rows),
                        unique_words=len(uw), uniq_in_vocab=int(sum(uflag)), uniq_oov=int(len(uw) - sum(uflag)),
                        pairs_both_in=int(pair_in.all(1).sum()), pairs_one_oov=int((pair_in.sum(1) == 1).sum()),
                        pairs_both_oov=int((pair_in.sum(1) == 0).sum()), sims=sims, gold=gold)
    res["__macro__"] = float(np.nanmean([v["rho"] for k, v in res.items() if k != "__macro__"]))
    return res

def train_model(sents, kind="ft", **kw):
    cfg = dict(sg=1, window=5, min_count=2, epochs=15, negative=10, min_n=2, max_n=6, bucket=200_000, sample=1e-4); cfg.update(kw)
    common = dict(vector_size=DIM, window=cfg["window"], min_count=cfg["min_count"], sg=cfg["sg"], negative=cfg["negative"],
                  sample=cfg["sample"], workers=NCPU, seed=SEED)
    m = FastText(min_n=cfg["min_n"], max_n=cfg["max_n"], bucket=cfg["bucket"], **common) if kind == "ft" else Word2Vec(**common)
    m.build_vocab(sents); m.train(sents, total_examples=m.corpus_count, epochs=cfg["epochs"])
    return m

def run_experiment(name, kind="ft", per_language=False, **kw):
    t0 = time.time()
    if per_language:
        models = {}
        for key in BENCH:
            sents = SENTS_BY_LANG.get(BENCH_CODE.get(key)) or ALL_SENTS
            models[key] = train_model(sents, kind, **kw)
        emb = Embedder(models)
    else:
        emb = Embedder(train_model(ALL_SENTS, kind, **kw), oov_zero=(kind == "w2v"))
    r = evaluate(emb)
    print(f"{name:<44s} macro rho = {r['__macro__']:.4f}  ({time.time()-t0:.0f}s)")
    return dict(name=name, kind=kind, per_language=per_language, kw=kw, emb=emb, res=r)

## 5. Experiments
All settings are trained from scratch on IndicCorpV2. The public benchmark is used as a **validation set** to pick among these few configurations; scores are never used to update embeddings.

In [7]:
EXPS = []
QUICK = True   # True: 4 trainings (~3-4x faster). False: full 9-experiment grid for the report.
EP = 10 if QUICK else 15
EXPS.append(run_experiment("E1 FastText skip-gram (default)", epochs=EP))
EXPS.append(run_experiment("E3 FastText SG, n-grams 3-6", min_n=3, max_n=6, epochs=EP))
if not QUICK:
    EXPS.append(run_experiment("E2 FastText CBOW", sg=0))
    EXPS.append(run_experiment("E4 FastText SG, n-grams 2-5", min_n=2, max_n=5))
    EXPS.append(run_experiment("E5 FastText SG, 30 epochs", epochs=30))
    EXPS.append(run_experiment("E6 FastText SG, window 10", window=10))
    EXPS.append(run_experiment("E7 FastText SG, min_count 1", min_count=1))
if SENTS_BY_LANG:
    EXPS.append(run_experiment("E8 Per-language FastText SG", per_language=True, epochs=EP))
EXPS.append(run_experiment("E9 Word2Vec SG baseline (OOV -> zero vec)", kind="w2v", epochs=EP))

exp_table = pd.DataFrame([{**{"experiment": e["name"]}, **{k: round(v["rho"], 4) for k, v in e["res"].items() if k != "__macro__"},
                           "macro": round(e["res"]["__macro__"], 4)} for e in EXPS])
exp_table

E1 FastText skip-gram (default)              macro rho = 0.1303  (1174s)
E3 FastText SG, n-grams 3-6                  macro rho = 0.0506  (1056s)
E8 Per-language FastText SG                  macro rho = 0.1753  (262s)
E9 Word2Vec SG baseline (OOV -> zero vec)    macro rho = -0.0034  (699s)


,experiment,Gujarati-WS,Hindi-WS,Marathi-WS,Punjabi-WS,Tamil-WS,Telugu-WS,Urdu-WS,macro
0,E1 FastText skip-gram (default),0.2402,0.1659,0.1789,0.1268,0.1650,0.0304,0.0051,0.1303
1,"E3 FastText SG, n-grams 3-6",-0.0628,0.0374,0.1508,0.1521,0.1801,-0.0684,-0.0349,0.0506
2,E8 Per-language FastText SG,0.2130,0.1860,-0.0892,0.2831,0.3327,0.1200,0.1816,0.1753
3,E9 Word2Vec SG baseline (OOV -> zero vec),-0.1377,0.0104,-0.0305,0.2136,0.0723,-0.0368,-0.1151,-0.0034


In [8]:
# choose the best subword model (the W2V baseline is excluded: it cannot handle OOV words)
cands = [e for e in EXPS if e["kind"] == "ft"]
BEST = max(cands, key=lambda e: e["res"]["__macro__"])
RES = BEST["res"]; EMB = BEST["emb"]
print("selected:", BEST["name"], "| macro Spearman:", round(RES["__macro__"], 4))
EMB.save(OUT/"model")

selected: E8 Per-language FastText SG | macro Spearman: 0.1753


## 6. Vocabulary coverage & OOV statistics

In [9]:
cov_rows = []
for k, v in RES.items():
    if k == "__macro__": continue
    cov_rows.append({"language": k, "pairs": v["n_pairs"], "unique words": v["unique_words"],
                     "in-vocab": v["uniq_in_vocab"], "OOV": v["uniq_oov"],
                     "OOV %": round(100 * v["uniq_oov"] / v["unique_words"], 1),
                     "pairs both in": v["pairs_both_in"], "pairs 1 OOV": v["pairs_one_oov"], "pairs both OOV": v["pairs_both_oov"],
                     "Spearman": round(v["rho"], 4)})
cov_table = pd.DataFrame(cov_rows)
print("Macro-average Spearman:", round(RES["__macro__"], 4))
cov_table

Macro-average Spearman: 0.1753


,language,pairs,unique words,in-vocab,OOV,OOV %,pairs both in,pairs 1 OOV,pairs both OOV,Spearman
0,Gujarati-WS,163,221,177,44,19.9,116,37,10,0.2130
1,Hindi-WS,235,318,256,62,19.5,163,52,20,0.1860
2,Marathi-WS,104,135,96,39,28.9,55,42,7,-0.0892
3,Punjabi-WS,143,206,173,33,16.0,106,33,4,0.2831
4,Tamil-WS,96,135,102,33,24.4,60,35,1,0.3327
5,Telugu-WS,111,160,108,52,32.5,56,45,10,0.1200
6,Urdu-WS,100,146,126,20,13.7,77,19,4,0.1816


## 7. Export embeddings: `(N, 2, 300)` `.npy` per language + vocabulary files

In [10]:
def export_all(emb, tag=""):
    manifest = {}
    for key, rows in BENCH.items():
        arr = np.stack([np.stack([emb.vec(a, key), emb.vec(b, key)]) for a, b, _ in rows]).astype(np.float32)
        assert arr.shape == (len(rows), 2, 300), arr.shape
        np.save(OUT/"embeddings"/f"{key}_embeddings.npy", arr)               # order == original pair order
        uw = list(dict.fromkeys(w for a, b, _ in rows for w in (a, b)))
        np.save(OUT/"vocab"/f"{key}_word_vectors.npy", np.stack([emb.vec(w, key) for w in uw]).astype(np.float32))
        json.dump({"language": key, "dim": 300, "word_to_row": {w: i for i, w in enumerate(uw)}, "vectors_file": f"{key}_word_vectors.npy",
                   "pair_rows": [[uw.index(a), uw.index(b)] for a, b, _ in rows]},
                  open(OUT/"vocab"/f"{key}_vocab.json", "w"), ensure_ascii=False)
        pd.DataFrame(rows, columns=["word1", "word2", "human_score"]).assign(cosine=RES[key]["sims"]).to_csv(OUT/"embeddings"/f"{key}_pairs.csv", index=False, encoding="utf-8")
        manifest[key] = list(arr.shape)
    json.dump(manifest, open(OUT/"embeddings"/"manifest.json", "w"), indent=1)
    return manifest
print(export_all(EMB))

{'Gujarati-WS': [163, 2, 300], 'Hindi-WS': [235, 2, 300], 'Marathi-WS': [104, 2, 300], 'Punjabi-WS': [143, 2, 300], 'Tamil-WS': [96, 2, 300], 'Telugu-WS': [111, 2, 300], 'Urdu-WS': [100, 2, 300]}


## 8. t-SNE (100 sampled words per language)

In [11]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.ft2font import FT2Font
from sklearn.manifold import TSNE

_font_cache = {}
def indic_font(sample):
    chars = {ord(c) for c in sample if not c.isspace()}
    for f in fm.fontManager.ttflist:
        try:
            if f.fname not in _font_cache: _font_cache[f.fname] = set(FT2Font(f.fname).get_charmap().keys())
            if chars <= _font_cache[f.fname]: return fm.FontProperties(fname=f.fname)
        except Exception: continue
    return None

TSNE_FIGS = {}
for key, rows in BENCH.items():
    uw = sorted(dict.fromkeys(w for a, b, _ in rows for w in (a, b)))
    rng = random.Random(SEED); words = rng.sample(uw, min(100, len(uw)))
    X = np.stack([EMB.vec(w, key) for w in words])
    Z = TSNE(n_components=2, perplexity=min(30, max(2, (len(words) - 1) // 3)), init="pca", random_state=SEED).fit_transform(X)
    fig, ax = plt.subplots(figsize=(6.2, 5))
    ax.scatter(Z[:, 0], Z[:, 1], s=14, alpha=.7)
    fp = indic_font("".join(words[:20]))
    if fp:
        for (x, y), w in zip(Z, words[:25]): ax.annotate(w, (x, y), fontproperties=fp, fontsize=7)
    ax.set_title(f"t-SNE of {len(words)} words - {key}"); ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout(); path = OUT/"figures"/f"tsne_{key}.png"; fig.savefig(path, dpi=150); plt.close(fig)
    TSNE_FIGS[key] = path
print("saved", len(TSNE_FIGS), "figures")
# Tip (Colab): `!apt-get -qq install fonts-noto-core` then restart & rerun to get readable Indic labels on the plots.

saved 7 figures


## 9. Error analysis: correctly / incorrectly predicted pairs

In [12]:
from scipy.stats import rankdata
ana = []
for key, rows in BENCH.items():
    v = RES[key]; n = len(rows)
    rm, rh = rankdata(v["sims"]) / n, rankdata(v["gold"]) / n
    for i, (a, b, s) in enumerate(rows):
        ana.append(dict(language=key, word1=a, word2=b, human=s, cosine=round(float(v["sims"][i]), 3),
                        human_pct=round(float(rh[i]), 2), model_pct=round(float(rm[i]), 2), rank_err=round(float(abs(rm[i] - rh[i])), 3),
                        oov=int(not EMB.in_vocab(a, key)) + int(not EMB.in_vocab(b, key))))
ana = pd.DataFrame(ana)
ana.to_csv(OUT/"analysis_all_pairs.csv", index=False, encoding="utf-8")
best_ex  = ana.sort_values("rank_err").groupby("language").head(1).sort_values("rank_err").head(5)
worst_ex = ana.sort_values("rank_err", ascending=False).groupby("language").head(1).sort_values("rank_err", ascending=False).head(5)
display(best_ex); display(worst_ex)

,language,word1,word2,human,cosine,human_pct,model_pct,rank_err,oov
2,Gujarati-WS,વાઘ,વાઘ,10.0,1.000,1.00,1.00,0.000,0
944,Urdu-WS,کریڈٹ,معلومات,2.0,0.550,0.18,0.18,0.000,0
681,Tamil-WS,பானம்,காது,0.0,0.569,0.04,0.04,0.000,1
399,Marathi-WS,वाघ,वाघ,10.0,1.000,1.00,1.00,0.000,0
522,Punjabi-WS,ਫੁੱਟਬਾਲ,ਫੁੱਟਬਾਲ,10.0,1.000,1.00,1.00,0.003,0


,language,word1,word2,human,cosine,human_pct,model_pct,rank_err,oov
884,Urdu-WS,منی,گہنا,9.500,0.336,1.00,0.01,0.990,1
211,Hindi-WS,पैसे,धन,9.200,0.546,1.00,0.04,0.957,0
419,Marathi-WS,पैसा,रोख,9.375,0.596,0.98,0.04,0.938,0
692,Tamil-WS,உணவு,சேவல்,10.000,0.622,0.98,0.09,0.885,0
541,Punjabi-WS,ਪੈਸਾ,ਕਰੰਸੀ,8.750,0.593,0.96,0.08,0.878,0


## 10. Inference API — embeddings for arbitrary words (incl. OOV) for the private test set

`embed_words(words, language=None)` -> `(len(words), 300)`; `embed_pairs(pairs, language=None)` -> `(N, 2, 300)`.
If the model is not in memory it is loaded from `fovl_work/submission/model/`.

In [13]:
def load_embedder(model_dir=OUT/"model"):
    meta = json.load(open(Path(model_dir)/"meta.json"))
    if meta["per_language"]:
        return Embedder({k: FastText.load(str(Path(model_dir)/f"{k}.model")) for k in meta["keys"]})
    return Embedder(FastText.load(str(Path(model_dir)/"joint.model")))

_EMB = None
def embed_words(words, language=None, model_dir=OUT/"model"):
    global _EMB
    if _EMB is None: _EMB = load_embedder(model_dir)
    return np.stack([_EMB.vec(w, language) for w in words]).astype(np.float32)

def embed_pairs(pairs, language=None, model_dir=OUT/"model"):
    flat = embed_words([w for p in pairs for w in p[:2]], language, model_dir)
    return flat.reshape(len(pairs), 2, DIM)

# sanity: reloaded model reproduces the exported arrays, and handles an unseen word
key0 = next(iter(BENCH)); rows0 = BENCH[key0][:5]
chk = embed_pairs(rows0, key0)
ref = np.load(OUT/"embeddings"/f"{key0}_embeddings.npy")[:5]
print("reload max abs diff:", float(np.abs(chk - ref).max()), "| shape:", chk.shape)
print("unseen word ->", embed_words(["ऽऽअज्ञातशब्दxyz"], key0).shape)

reload max abs diff: 0.0 | shape: (5, 2, 300)
unseen word -> (1, 300)


## 11. Report (DOCX -> PDF) and submission ZIP

In [14]:
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH

def build_report(path):
    d = Document()
    for s in d.sections: s.left_margin = s.right_margin = Inches(1); s.top_margin = s.bottom_margin = Inches(0.9)
    st = d.styles["Normal"]; st.font.name = "Calibri"; st.font.size = Pt(11)
    def H(t, l=1): d.add_heading(t, level=l)
    def P(t): return d.add_paragraph(t)
    def table(df, size=8.5):
        t = d.add_table(rows=1, cols=len(df.columns)); t.style = "Light Grid Accent 1"
        for i, c in enumerate(df.columns): t.rows[0].cells[i].text = str(c)
        for _, r in df.iterrows():
            cells = t.add_row().cells
            for i, v in enumerate(r): cells[i].text = str(v)
        for row in t.rows:
            for c in row.cells:
                for p in c.paragraphs:
                    for run in p.runs: run.font.size = Pt(size)
        d.add_paragraph()
    langs = [k for k in RES if k != "__macro__"]
    title = d.add_heading("FOVL Assignment: Multilingual Word Embeddings for Indian Languages", 0)
    P(f"{STUDENT_NAME} ({ROLL_NO})  |  Foundations of Vision & Language")

    H("1. Introduction")
    P(f"The goal is to learn 300-dimensional word embeddings for Indian languages using only the IndicCorpV2 corpus, and to evaluate them intrinsically on the word-similarity benchmarks of Akhtar et al. (2017). "
      f"We evaluate on {len(langs)} benchmark sets ({', '.join(langs)}). Indian languages are morphologically rich, so many word forms are rare or unseen in a small corpus; "
      "we therefore use a subword-aware model so that every word, including out-of-vocabulary (OOV) words, receives a meaningful vector.")

    H("2. Methodology")
    cfg = BEST["kw"]
    P(f"Data. The corpus is {len(ALL_SENTS):,} sentences / {tok_total:,} tokens from IndicCorpV2 (5k sample). Text is NFC-normalised, lower-cased, zero-width characters, punctuation (including the danda) and digits are removed, and sentences are split on newlines, danda and sentence punctuation.")
    P(f"Model. Selected model: {BEST['name']} (gensim FastText, vector size 300, "
      + ("one model per language" if BEST["per_language"] else "a single model trained jointly on all languages")
      + f"; settings overriding defaults: {cfg if cfg else 'defaults'}). Defaults: skip-gram, window 5, negative sampling with 10 negatives, min_count 2, 15 epochs, character n-grams 2-6 hashed into 200k buckets, sub-sampling 1e-4, seed {SEED}. All weights are trained from scratch; no pre-trained model or external corpus is used.")
    P("OOV handling. A word's vector is the average of its own vector (if in vocabulary) and the vectors of its character n-grams. An unseen word is represented by the sum/average of its n-gram vectors, so inflected or compounded forms that share n-grams with training words get nearby vectors. Benchmark entries that tokenise to several tokens are averaged. No word pair is dropped.")
    P("Evaluation. For every pair we compute cosine similarity and report Spearman's rho against the human scores, per language and macro-averaged. Human scores are never used to train the embeddings; they were used only to compare a small number of hyper-parameter configurations (Section 5) as a validation set.")

    H("3. t-SNE Visualization")
    P("Each plot shows t-SNE (perplexity <= 30, PCA init) of 100 randomly sampled benchmark words per language (all words if fewer than 100 are available).")
    keys = list(TSNE_FIGS)
    for i in range(0, len(keys), 2):
        t = d.add_table(rows=2, cols=2)
        for j, k in enumerate(keys[i:i+2]):
            t.cell(0, j).paragraphs[0].add_run().add_picture(str(TSNE_FIGS[k]), width=Inches(3.1))
            t.cell(1, j).text = f"Figure: {k}"

    H("4. Results")
    P(f"Macro-average Spearman correlation over {len(langs)} languages: {RES['__macro__']:.4f}.")
    table(cov_table[["language", "pairs", "Spearman"]].assign(**{"cosine mean": [round(float(np.mean(RES[k]['sims'])), 3) for k in langs]}))
    P("Vocabulary coverage and OOV statistics of the benchmark words per language:")
    table(cov_table.drop(columns=["Spearman"]), 7.5)

    H("5. Experiments")
    P("All configurations are trained from scratch on IndicCorpV2 and evaluated by Spearman's rho on the public benchmark (macro = mean over languages). E9 is a word-level Word2Vec baseline without subwords (OOV words get a zero vector, cosine 0) that quantifies the benefit of the OOV strategy; it is not eligible for submission.")
    table(exp_table, 7.5)
    P(f"Selected: {BEST['name']} (macro {RES['__macro__']:.4f}).")

    H("6. Analysis")
    P("Pairs whose normalised rank under the model is close to their rank under human judgement are 'correct'; large rank differences are 'incorrect'. Percentiles are rank / N within the language.")
    ex = pd.concat([best_ex.assign(verdict="correct"), worst_ex.assign(verdict="incorrect")])[["verdict", "language", "word1", "word2", "human", "cosine", "human_pct", "model_pct", "oov"]]
    table(ex.rename(columns={"oov": "#OOV words"}), 8)
    P("Observations: pairs sharing morphology or surface n-grams (inflections of the same stem) tend to receive high cosine similarity regardless of the human rating, which explains many errors; rare words whose vectors rely mostly on n-grams are less reliable than frequent in-vocabulary words. Pairs of frequent, topically related words are ranked best.")
    P("Full per-pair results are provided in analysis_all_pairs.csv.")

    H("7. Discussion")
    P("Design choices. FastText was chosen over Word2Vec/GloVe because rich morphology and a small (5k-sample) corpus produce many rare and unseen forms; subword n-grams give every word a vector and share statistical strength across inflections. A joint multilingual model is simple and the scripts rarely overlap, so cross-language interference is small; per-language models were also tried (E8). Hyper-parameters were chosen from a small grid using the public benchmark as validation.")
    P("Limitations. (1) The corpus is small, so absolute correlations are modest and vary with the random seed and multi-threaded training order. (2) Hashing n-grams into a limited number of buckets introduces collisions. (3) Benchmarks are small, so Spearman estimates have wide confidence intervals; the private test set may rank configurations differently. (4) Embeddings are static: polysemy and context are not modelled. (5) Orthographic variation (Unicode forms, spelling variants) can cause avoidable OOVs; we apply only NFC normalisation. (6) Our hyper-parameter selection on the public benchmark may slightly over-fit it.")
    P("Disclosure. No pre-trained models or external datasets were used. Tools: Python, gensim (FastText, Word2Vec), NumPy, pandas, SciPy (Spearman), scikit-learn (t-SNE), matplotlib, python-docx. Data: IndicCorpV2-5k (training) and Akhtar et al. word-similarity datasets (evaluation only).")

    H("8. References")
    for r in ["Akhtar, S. S., Gupta, A., Vajpayee, A., Srivastava, A., Shrivastava, M. (2017). Word similarity datasets for Indian languages: Annotation and baseline systems. Linguistic Annotation Workshop (LAW XI).",
              "Bojanowski, P., Grave, E., Joulin, A., Mikolov, T. (2017). Enriching word vectors with subword information. TACL 5.",
              "Mikolov, T., Chen, K., Corrado, G., Dean, J. (2013). Efficient estimation of word representations in vector space. arXiv:1301.3781.",
              "Pennington, J., Socher, R., Manning, C. (2014). GloVe: Global vectors for word representation. EMNLP.",
              "Doddapaneni, S. et al. (2023). Towards leaving no Indic language behind: Building monolingual corpora, benchmark and models for Indic languages. ACL (IndicCorp v2).",
              "Rehurek, R., Sojka, P. (2010). Software framework for topic modelling with large corpora. LREC Workshop (gensim).",
              "van der Maaten, L., Hinton, G. (2008). Visualizing data using t-SNE. JMLR 9."]:
        d.add_paragraph(r, style="List Number")
    d.save(path)

REPORT_DOCX = OUT/"report.docx"; build_report(REPORT_DOCX); print("report ->", REPORT_DOCX)
soffice = shutil.which("soffice") or shutil.which("libreoffice")
if soffice:
    subprocess.run([soffice, "--headless", "--convert-to", "pdf", "--outdir", str(OUT), str(REPORT_DOCX)], capture_output=True)
    print("PDF ->", OUT/"report.pdf", (OUT/"report.pdf").exists())
else:
    print("LibreOffice not found: open report.docx in Word/Docs and 'Save as PDF' (Colab: !apt-get -qq install libreoffice-writer, then rerun this cell).")

report -> fovl_work/submission/report.docx
LibreOffice not found: open report.docx in Word/Docs and 'Save as PDF' (Colab: !apt-get -qq install libreoffice-writer, then rerun this cell).


In [15]:
# ---------------- package the submission ZIP ----------------
zip_path = WORK/"FOVL_submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if not p.is_file(): continue
        if "model" in p.relative_to(OUT).parts and not INCLUDE_MODEL_IN_ZIP: continue
        if p.name == "report.docx": continue
        z.write(p, p.relative_to(OUT))
    nbs = [f for f in Path(".").glob("*.ipynb")]
    for nbp in nbs: z.write(nbp, nbp.name)         # notebook (save it first so the latest version is included)
print("wrote", zip_path, f"({zip_path.stat().st_size/1e6:.1f} MB)")
print("Remember to add the .ipynb to the ZIP if it was not found in the working directory, and check report.pdf.")

wrote fovl_work/FOVL_submission.zip (3.3 MB)
Remember to add the .ipynb to the ZIP if it was not found in the working directory, and check report.pdf.
